# Datathon 2026 — MULTIMODAL (XLM-R-large + tabular NN) | BIZIM folds.parquet repeat-0

**Amac:** GBDT/linear noise-floor'u FARKLI bir fonksiyon sinifiyla (neural multimodal) kirma denemesi. Tek mesru atis.

**KRITIK — fold sozlesmesi:** Bu notebook bizim kanonik `data/folds.parquet` dosyasinin **repeat==0** (5-fold) atamasini kullanir. Onceki notebook'lar ayri bir `folds.npy` kullaniyordu; o cikti satır-hizasız -> KULLANILMAZ. Burada `oof_mm[i]` = i'nin repeat-0'da val oldugu fold'un tahmini; satir sirasi train.csv student_id sirasi (mevcut `artifacts/oof_*.npy` ile birebir hizali).

**Runtime:** GPU (A100 ideal; XLM-R-large 2.2GB + bf16). Cell 2'de **YUKLE:** `train.csv`, `test_x.csv`, `folds.parquet`.

**Cikti (Cell son):** `oof_mm.npy` (10000,) + `test_mm.npy` (10000,) -> indir, repo'da `artifacts/` icine koy. Sonra lokalde `python src/mm_blend.py` + `python src/ensemble.py` ile GATED blend.

**Mimari (prompt):** text branch xlm-roberta-large mean+CLS-pooled, fold-ici fine-tune (MAX_LEN=192, BATCH=16, EPOCHS=3, LR_bert=1e-5, LR_head=1e-3, AdamW, bf16, warmup); tabular branch fold-safe 82-feat matris -> per-fold median-impute + StandardScaler (yalnız fold-train'e fit) -> MLP(256->128); fusion -> head -> z-score hedef, tahmin xystd+ymean, clip[0,100]. **HEDEF-ENCODING YOK.**

**Reproducibility:** seed=42 (torch/numpy/cuda). Neural/GPU bit-deterministik DEGIL (cuDNN/atomik) -> SUB-2'ye girerse repro 'belgelenmis tolerans' (bit-ayni degil). SUB-1 (catboost_full) dokunulmaz, bit-reproducible kalır.

In [ ]:
!pip -q install transformers==4.44.2 accelerate sentencepiece pyarrow

In [ ]:
from google.colab import files
up = files.upload()  # YUKLE: train.csv, test_x.csv, folds.parquet

In [ ]:
# === Cell 3: SABITLER + FOLD-SAFE TABULAR MATRIS (src/cv.py + src/tabular_mm.py ile BIREBIR) ===
# Kolon rolleri TEK kaynak: repo src/cv.py. Burada (Colab izole olsun diye) INLINE kopyalandi;
# repo ile ayni -> ayni matris. one-hot + missing-flag HEDEF-BAGIMSIZ (fold-safe; global hesap
# sizinti DEGIL). Tek hedef-bagimli adim (impute+scale) Cell 5'te PER-FOLD fold-train'e fit edilir.
import numpy as np, pandas as pd, json
from pandas.api.types import CategoricalDtype

SEED = 42
TARGET = "career_success_score"; TEXT = "mentor_feedback_text"; ID = "student_id"
CATEGORICAL_COLS = ("department","university_tier","target_role","hobby","preferred_social_media_platform")
YEAR_COLS = ("application_year","graduation_year")
NA_COLS = ("internship_duration_months","english_exam_score","github_avg_stars",
           "open_source_contribution_count","hr_interview_score","linkedin_profile_score","portfolio_score")
CLIP_LO, CLIP_HI = 0.0, 100.0
RECENCY_COL = "graduation_year"

train = pd.read_csv("train.csv", encoding="utf-8-sig")
test  = pd.read_csv("test_x.csv", encoding="utf-8-sig")
print("train", train.shape, "test", test.shape)
assert train[ID].iloc[0] == "STU_000001" and test[ID].iloc[0] == "STU_010001", "ID sirasi beklenmedik."

def numeric_feature_columns(df):
    drop = {ID, TARGET, TEXT, *CATEGORICAL_COLS, *YEAR_COLS}
    return [c for c in df.columns if c not in drop]

def structured_cat_dtypes(tr):
    return {c: CategoricalDtype(categories=sorted(tr[c].astype(str).unique())) for c in CATEGORICAL_COLS}

def build_tabular_matrix(tr, te):
    """HAM (olceksiz/imputesiz) fold-safe NN matrisi. NaN sayisal/yil'da KORUNUR. -> (Xtr,Xte,cols)."""
    num_cols = numeric_feature_columns(tr); year_cols = list(YEAR_COLS); na_cols = list(NA_COLS)
    cat_dtypes = structured_cat_dtypes(tr)
    def _one(df):
        parts = [df[num_cols].astype("float32").reset_index(drop=True),
                 df[year_cols].astype("float32").reset_index(drop=True)]
        fl = df[na_cols].isna().astype("float32"); fl.columns = [f"{c}_missing" for c in na_cols]
        parts.append(fl.reset_index(drop=True))
        for c in CATEGORICAL_COLS:
            s = df[c].astype(str).astype(cat_dtypes[c])
            d = pd.get_dummies(s, prefix=c, dtype="float32")
            expected = [f"{c}_{lvl}" for lvl in cat_dtypes[c].categories]
            d = d.reindex(columns=expected, fill_value=np.float32(0.0))
            parts.append(d.reset_index(drop=True))
        out = pd.concat(parts, axis=1); return out, list(out.columns)
    Xtr_df, cols = _one(tr); Xte_df, cols_te = _one(te)
    assert cols == cols_te, "train/test one-hot kolonlari ayni degil."
    return Xtr_df.to_numpy(np.float32), Xte_df.to_numpy(np.float32), cols

TAB_TR, TAB_TE, TAB_COLS = build_tabular_matrix(train, test)
y = train[TARGET].values.astype(np.float32)
txt_tr = train[TEXT].fillna("").tolist(); txt_te = test[TEXT].fillna("").tolist()
print(f"tabular: {TAB_TR.shape} (37 num + 2 year + 7 flag + {TAB_TR.shape[1]-46} one-hot)  NaN-sutun={int(np.isnan(TAB_TR).any(0).sum())}")

In [ ]:
# === Cell 4: BIZIM folds.parquet repeat-0 fold atamasi (student_id sirasinda) ===
folds = pd.read_parquet("folds.parquet")
assert {"student_id","repeat","fold"}.issubset(folds.columns), "folds.parquet semasi beklenmedik."
pos = {sid: i for i, sid in enumerate(train[ID].values)}
fr0 = folds[folds["repeat"] == 0]
fold_of = np.full(len(train), -1, dtype=np.int64)
for sid, f in zip(fr0["student_id"].values, fr0["fold"].values):
    fold_of[pos[sid]] = int(f)
assert (fold_of >= 0).all(), "repeat-0: bazi satirlarin fold'u yok."
N_SPLITS = int(fold_of.max()) + 1
FOLDS = [(np.where(fold_of != f)[0], np.where(fold_of == f)[0]) for f in range(N_SPLITS)]
print(f"repeat-0: {N_SPLITS} fold, val sayilari = {[len(v) for _,v in FOLDS]}")
assert sum(len(v) for _,v in FOLDS) == len(train), "OOF kapsami bozuk (her satir 1x val olmali)."

In [ ]:
# === Cell 5: MULTIMODAL model + fold-safe egitim (XLM-R-large mean+CLS + tabular MLP) ===
import torch, torch.nn as nn, os, random
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

MODEL_NAME = "xlm-roberta-large"
MAX_LEN, BATCH, EPOCHS = 192, 16, 3
LR_BERT, LR_HEAD = 1e-5, 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP = torch.bfloat16 if USE_BF16 else torch.float16

# --- seed (bit-deterministik DEGIL; belgelenmis tolerans) ---
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("device:", DEVICE, "| model:", MODEL_NAME, "| amp:", AMP)

ymean, ystd = float(y.mean()), float(y.std())  # NOT: per-fold yeniden hesaplanir (asagida)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class MMData(Dataset):
    def __init__(self, texts, tab, tgt=None, ym=0.0, ys=1.0):
        self.t=texts; self.tab=tab; self.y=tgt; self.ym=ym; self.ys=ys
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        e = tok(self.t[i], truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        it = {"input_ids": e["input_ids"].squeeze(0), "attention_mask": e["attention_mask"].squeeze(0),
              "tab": torch.tensor(self.tab[i], dtype=torch.float32)}
        if self.y is not None:
            it["label"] = torch.tensor((self.y[i]-self.ym)/self.ys, dtype=torch.float32)
        return it

class MultiModal(nn.Module):
    def __init__(self, ntab):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME); h = self.bert.config.hidden_size
        self.tab = nn.Sequential(nn.Linear(ntab,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(256,128), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(2*h+128,256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256,1))
    def forward(self, input_ids, attention_mask, tab):
        hs = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        mean = (hs*m).sum(1)/m.sum(1).clamp(min=1e-9); cls = hs[:,0]
        return self.head(torch.cat([mean, cls, self.tab(tab)], dim=1)).squeeze(-1)

@torch.no_grad()
def predict(model, texts, tab, ym, ys):
    model.eval(); out=[]
    for b in DataLoader(MMData(texts, tab), batch_size=64):
        ids=b["input_ids"].to(DEVICE); am=b["attention_mask"].to(DEVICE); tb=b["tab"].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=AMP): p = model(ids, am, tb)
        out.append(p.float().cpu().numpy())
    return np.concatenate(out)*ys + ym

oof = np.zeros(len(train), dtype=np.float64); test_pred = np.zeros(len(test), dtype=np.float64)
for f,(tr_idx, va_idx) in enumerate(FOLDS):
    print(f"\n== Fold {f+1}/{N_SPLITS} ==")
    # --- FOLD-SAFE tabular: impute (fold-train medyani) + StandardScaler (yalnız fold-train'e fit) ---
    med = np.nanmedian(TAB_TR[tr_idx], axis=0)            # fold-train medyani (test/val'e bakmaz)
    med = np.where(np.isnan(med), 0.0, med)               # tum-NaN sutun (olmamali) -> 0
    def _imp(M):
        M = M.copy(); idx = np.where(np.isnan(M)); M[idx] = np.take(med, idx[1]); return M
    sc = StandardScaler().fit(_imp(TAB_TR[tr_idx]))
    Xtr = sc.transform(_imp(TAB_TR[tr_idx])).astype(np.float32)
    Xva = sc.transform(_imp(TAB_TR[va_idx])).astype(np.float32)
    Xte = sc.transform(_imp(TAB_TE)).astype(np.float32)
    # --- FOLD-SAFE hedef z-score: yalnız fold-train y'sinden ---
    ym, ys = float(y[tr_idx].mean()), float(y[tr_idx].std())

    model = MultiModal(Xtr.shape[1]).to(DEVICE)
    dl = DataLoader(MMData([txt_tr[i] for i in tr_idx], Xtr, y[tr_idx], ym, ys),
                    batch_size=BATCH, shuffle=True, drop_last=True)
    bp=[p for n,p in model.named_parameters() if n.startswith("bert.")]
    hp=[p for n,p in model.named_parameters() if not n.startswith("bert.")]
    opt = torch.optim.AdamW([{"params":bp,"lr":LR_BERT},{"params":hp,"lr":LR_HEAD}], weight_decay=0.01)
    tot = len(dl)*EPOCHS; sch = get_linear_schedule_with_warmup(opt, int(0.1*tot), tot)
    scaler = torch.cuda.amp.GradScaler(enabled=not USE_BF16)
    for ep in range(EPOCHS):
        model.train()
        for b in dl:
            opt.zero_grad()
            ids=b["input_ids"].to(DEVICE); am=b["attention_mask"].to(DEVICE)
            tb=b["tab"].to(DEVICE); lab=b["label"].to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=AMP):
                loss = nn.functional.mse_loss(model(ids, am, tb), lab)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
        vp = np.clip(predict(model, [txt_tr[i] for i in va_idx], Xva, ym, ys), CLIP_LO, CLIP_HI)
        print(f"   epoch {ep+1} val MSE={np.mean((y[va_idx]-vp)**2):.4f}")
    oof[va_idx] = np.clip(predict(model, [txt_tr[i] for i in va_idx], Xva, ym, ys), CLIP_LO, CLIP_HI)
    test_pred  += np.clip(predict(model, txt_te, Xte, ym, ys), CLIP_LO, CLIP_HI) / N_SPLITS
    del model; torch.cuda.empty_cache()

oof = np.clip(oof, CLIP_LO, CLIP_HI); test_pred = np.clip(test_pred, CLIP_LO, CLIP_HI)
assert (np.bincount(fold_of).sum() == len(train))  # her satir tam 1x val (repeat-0)
print(f"\n=== MM unweighted-OOF (repeat-0, bizim fold) = {np.mean((y-oof)**2):.4f} ===")

In [ ]:
# === Cell 6: standalone rw-OOF (recency-weighted; KARAR METRIGINE en yakin tek-tablo gostergesi) ===
# cv.recency_weights ile BIREBIR: w = P_test(graduation_year)/P_train(graduation_year), mean-norm.
tr_g = train[RECENCY_COL].value_counts(normalize=True)
te_g = test[RECENCY_COL].value_counts(normalize=True)
w = (train[RECENCY_COL].map(te_g).fillna(0.0) / train[RECENCY_COL].map(tr_g)).to_numpy(float)
w = w / w.mean()
rw = float(np.sum(w*(y-oof)**2)/np.sum(w))
print(f"MM standalone:  unweighted-OOF = {np.mean((y-oof)**2):.4f}   recency-weighted-OOF = {rw:.4f}")
print("  (referans tek-model rw-OOF: catboost_full 86.41 / lgbm_full 87.27 / e5_ridge standalone 158.46)")
print("  (mevcut blend nested rw-OOF = 84.85 ; rakip public ~83.18)")
print("  NOT: bu repeat-0 standalone rw-OOF; NIHAI KARAR lokalde src/ensemble.py NESTED rw-OOF + paired-test.")

In [ ]:
# === Cell 7: artefakt yaz + indir (repo artifacts/ icine koy) ===
assert oof.shape == (len(train),) and test_pred.shape == (len(test),)
assert np.isfinite(oof).all() and np.isfinite(test_pred).all()
assert oof.min()>=0 and oof.max()<=100 and test_pred.min()>=0 and test_pred.max()<=100
np.save("oof_mm.npy", oof.astype(np.float64))
np.save("test_mm.npy", test_pred.astype(np.float64))
print("YAZILDI: oof_mm.npy", oof.shape, "| test_mm.npy", test_pred.shape)
from google.colab import files
files.download("oof_mm.npy"); files.download("test_mm.npy")